# TinyLlama LoRA to GGUF on Colab

Use this notebook for supervised TinyLlama LoRA training, merge, GGUF conversion, and artifact export. A T4 is acceptable with conservative limits. For full-dataset runs, prefer L4, A10, A100, or another larger CUDA GPU.

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime, timezone

REPO_URL = 'https://github.com/ashioyajotham/weather_forecasting_lora.git'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_ROOT = '/content/drive/MyDrive/weather_forecasting_lora_runs'
RUN_DIR = f'{DRIVE_ROOT}/tinyllama_gguf_{RUN_ID}'
os.environ['RUN_DIR'] = RUN_DIR
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR=', RUN_DIR)

In [ ]:
!rm -rf /content/weather_forecasting_lora
!git clone {REPO_URL} /content/weather_forecasting_lora
%cd /content/weather_forecasting_lora
!git rev-parse HEAD

In [ ]:
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install bitsandbytes

In [ ]:
import os

os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WEATHER_LORA_REPORT_TO'] = 'none'

# T4-safe default. Increase after one clean run.
os.environ['WEATHER_LORA_MAX_SAMPLES'] = os.environ.get('WEATHER_LORA_MAX_SAMPLES', '1000')
print('WEATHER_LORA_MAX_SAMPLES=', os.environ['WEATHER_LORA_MAX_SAMPLES'])

In [ ]:
!python scripts/smoke_check.py
!python scripts/eval_smoke.py
!python scripts/generation_quality_eval.py --write-report "$RUN_DIR/generation_quality_baseline.json"

In [ ]:
!python train_lora_peft.py 2>&1 | tee "$RUN_DIR/train_lora_peft.log"

In [ ]:
!python merge_lora.py 2>&1 | tee "$RUN_DIR/merge_lora.log"

## GGUF conversion

Conversion is attempted in Colab. If llama.cpp dependencies fail on the runtime, export `models/weather-merged` and convert locally with the same command.

In [ ]:
!rm -rf /content/llama.cpp
!git clone https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!python -m pip install -r /content/llama.cpp/requirements.txt
!mkdir -p models/gguf
!python /content/llama.cpp/convert_hf_to_gguf.py models/weather-merged --outfile models/gguf/weather-tinyllama.gguf --outtype f16 2>&1 | tee "$RUN_DIR/convert_gguf.log"

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

paths = {
    'adapter': Path('models/weather-lora-peft/lora_adapter'),
    'merged': Path('models/weather-merged'),
    'gguf': Path('models/gguf/weather-tinyllama.gguf'),
}

for name, path in paths.items():
    target = Path(RUN_DIR) / name
    if path.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(path, target)
    elif path.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
    else:
        print('missing', name, path)

def path_size(path: Path) -> int:
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())

manifest = {
    'run_id': RUN_ID,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'gpu': subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1"),
    'weather_lora_max_samples': os.environ.get('WEATHER_LORA_MAX_SAMPLES'),
    'artifacts': {name: {'path': str(Path(RUN_DIR) / name), 'bytes': path_size(Path(RUN_DIR) / name)} for name in paths},
}
manifest_path = Path(RUN_DIR) / 'manifest_tinyllama_gguf.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## Local import after Colab

Copy the Drive `adapter`, `merged`, and `gguf` artifacts into local `models/weather-lora-peft/lora_adapter`, `models/weather-merged`, and `models/gguf/weather-tinyllama.gguf`. Then run local smoke checks and CLI/FastAPI inference before treating the run as accepted.